# Run ALL Archive JSON Content Tests

Runs the 7 segment test notebooks in sequence (FPA, FTA, UTA, BAIL, SBAIL, JOH, TD) and prints a single summary.

- Each child notebook writes its own results to `test_automation_runs2` / `test_automation_results2`.
- This notebook captures elapsed time and child-status (OK / ERROR / TIMEOUT) per segment.
- Final summary cell queries `test_automation_runs2` for the latest run per segment and reports PASS / FAIL / NO_DATA counts.

Set `TIMEOUT_SEC` below if any segment is expected to take longer than the default 2 hours.

In [0]:
TIMEOUT_SEC = 7200   # per-notebook timeout (2 hours)
SAMPLE_SIZE = "20"   # passed to each child notebook

NOTEBOOKS = [
    ("1_FPA_JSON_Content_TESTS",   "archive_fpa_json"),
    ("2_FTA_JSON_Content_TESTS",   "archive_fta_json"),
    ("3_UTA_JSON_Content_TESTS",   "archive_uta_json"),
    ("4_BAIL_JSON_Content_TESTS",  "archive_bail_json"),
    ("5_SBAIL_JSON_Content_TESTS", "archive_sbail_json"),
    ("6_JOH_JSON_Content_TESTS",   "archive_joh_json"),
    ("7_TD_JSON_Content_TESTS",    "archive_td_json"),
]

In [0]:
from datetime import datetime
import time

results = []
overall_start = time.time()

for nb_name, state in NOTEBOOKS:
    print(f"\n{'='*70}")
    print(f"  RUN: {nb_name}  (state={state})")
    print(f"{'='*70}")
    start = time.time()
    status, message = "OK", ""
    try:
        rv = dbutils.notebook.run(nb_name, TIMEOUT_SEC, {"sample_size": SAMPLE_SIZE})
        message = (rv or "")[:300]
    except Exception as e:
        status = "ERROR"
        message = f"{type(e).__name__}: {str(e)[:400]}"
        print(f"  !! {status}: {message}")
    elapsed = time.time() - start
    results.append({"notebook": nb_name, "state": state, "status": status,
                    "elapsed_s": round(elapsed, 1), "message": message})
    print(f"  -> {status} in {elapsed:.1f}s")

overall_elapsed = time.time() - overall_start
print(f"\n{'='*70}")
print(f"  All {len(NOTEBOOKS)} notebooks finished. Total elapsed: {overall_elapsed:.1f}s")
print(f"{'='*70}")

In [0]:
# Per-notebook child status summary
print(f"\n{'='*70}")
print(f"  CHILD-NOTEBOOK STATUS")
print(f"{'='*70}")
print(f"  {'Notebook':<32} {'State':<22} {'Result':<8} {'Elapsed':>10}")
print(f"  {'-'*32} {'-'*22} {'-'*8} {'-'*10}")
for r in results:
    print(f"  {r['notebook']:<32} {r['state']:<22} {r['status']:<8} {r['elapsed_s']:>8.1f}s")

In [0]:
# Pull the latest run per segment from test_automation_runs2 + summarise PASS/FAIL/NO_DATA counts.
from pyspark.sql import functions as F
from pyspark.sql.functions import col

states = [s for _, s in NOTEBOOKS]
try:
    runs_df = spark.table("test_automation_runs2")
    latest_per_state = (runs_df
        .filter(col("state_under_test").isin(*states))
        .groupBy("state_under_test")
        .agg(F.max("run_start_datetime").alias("latest_start")))
    latest_runs = (runs_df.alias("r")
        .join(latest_per_state.alias("l"),
              (col("r.state_under_test") == col("l.state_under_test")) &
              (col("r.run_start_datetime") == col("l.latest_start")),
              "inner")
        .select("r.run_id", "r.state_under_test", "r.run_start_datetime", "r.run_status"))
    latest_run_ids = [r.run_id for r in latest_runs.collect()]

    results_df = spark.table("test_automation_results2")
    counts = (results_df
        .filter(col("run_id").isin(*latest_run_ids))
        .groupBy("test_from_state", "status").count()
        .groupBy("test_from_state")
        .pivot("status", ["PASS", "FAIL", "ERROR", "NO_DATA"])
        .agg(F.first("count"))
        .na.fill(0)
        .orderBy("test_from_state"))

    print(f"\n{'='*70}")
    print(f"  LATEST RUN: TEST COUNTS PER SEGMENT")
    print(f"{'='*70}")
    counts.show(50, truncate=False)
except Exception as e:
    print(f"Could not summarise results tables: {e}")

In [0]:
# Combined evidence file across all 7 segments: CSV + XLSX + HTML in one folder.
# Pulls every result row from the latest run_id per state and writes them side-by-side.
import os
import pandas as pd
from datetime import datetime as _dt

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
combined_root = f"/Workspace/Users/{run_user}/Results/Archive_RUN_ALL_Tests"
os.makedirs(combined_root, exist_ok=True)
combined_folder = os.path.join(combined_root, f"{_dt.utcnow().strftime('%Y%m%d_%H%M%S')}-RUN_ALL_COMBINED")
os.makedirs(combined_folder, exist_ok=True)

try:
    combined_df = (results_df
        .filter(col("run_id").isin(*latest_run_ids))
        .select("test_from_state", "test_name", "test_field", "status", "message", "description")
        .orderBy("test_from_state", "test_name", "test_field"))
    pdf = combined_df.toPandas()

    # CSV
    csv_path = os.path.join(combined_folder, "all_segments_results.csv")
    pdf.to_csv(csv_path, index=False)

    # XLSX with status colouring
    xlsx_path = os.path.join(combined_folder, "all_segments_results.xlsx")
    try:
        STATUS_COLOURS = {
            "PASS":    "background-color:#d5f5e3;color:#27ae60;font-weight:bold;",
            "FAIL":    "background-color:#fadbd8;color:#e74c3c;font-weight:bold;",
            "NO_DATA": "background-color:#fef9e7;color:#f39c12;",
            "ERROR":   "background-color:#fdecea;color:#922b21;font-weight:bold;",
        }
        def _style(v): return STATUS_COLOURS.get(str(v).upper(), "")
        pdf.style.applymap(_style, subset=["status"]).to_excel(xlsx_path, index=False, engine="openpyxl")
    except Exception:
        pdf.to_excel(xlsx_path, index=False, engine="openpyxl")

    # HTML with a per-segment count header at the top
    per_segment_counts = (pdf.groupby(["test_from_state", "status"]).size()
                              .unstack(fill_value=0)
                              .reindex(columns=["PASS", "FAIL", "ERROR", "NO_DATA"], fill_value=0))
    html_path = os.path.join(combined_folder, "all_segments_results.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("<html><body style='font-family:Arial;'>")
        f.write(f"<h2>Archive JSON Content Tests — Combined Run ({_dt.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')})</h2>")
        f.write("<h3>Per-segment status counts</h3>")
        f.write(per_segment_counts.to_html())
        f.write("<h3>All test results</h3>")
        try:
            f.write(pdf.style.applymap(_style, subset=["status"]).to_html())
        except Exception:
            f.write(pdf.to_html(index=False))
        f.write("</body></html>")

    print(f"\n{'='*70}")
    print(f"  COMBINED EVIDENCE FILES ({len(pdf)} rows across {pdf['test_from_state'].nunique()} segments)")
    print(f"{'='*70}")
    print(f"  CSV:  {csv_path}")
    print(f"  XLSX: {xlsx_path}")
    print(f"  HTML: {html_path}")
except Exception as e:
    print(f"Could not write combined evidence file: {e}")

In [0]:
# Final pass/fail banner: ERROR if any child failed; FAIL if any segment has any FAIL; PASS otherwise.
child_errors = [r for r in results if r["status"] != "OK"]
segment_fails = []
try:
    fails = (spark.table("test_automation_results2")
        .filter(col("run_id").isin(*latest_run_ids))
        .filter(col("status") == "FAIL")
        .groupBy("test_from_state").count().collect())
    segment_fails = [(r["test_from_state"], r["count"]) for r in fails]
except Exception:
    pass

print(f"\n{'='*70}")
if child_errors:
    print(f"  OVERALL: ERROR  ({len(child_errors)} child notebook(s) failed to complete)")
elif segment_fails:
    total_f = sum(c for _, c in segment_fails)
    print(f"  OVERALL: FAIL  ({total_f} test failure(s) across {len(segment_fails)} segment(s))")
    for s, c in segment_fails:
        print(f"     {s}: {c} FAIL")
else:
    print(f"  OVERALL: PASS  (all segments completed, no test failures)")
print(f"{'='*70}")